<a href="https://colab.research.google.com/github/Evgeniya-Ryasnova/generative-mocap/blob/master/generative_mocap_reproduction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Generative MoCap Reproduction Study

## 0. Reproduction Notes

### Goal

* Retrain cGAN models.
* Reproduce the results from the original paper.

### Environment

* Google Colab
* NVIDIA Tesla T4
* Python 3.12
* PyTorch 2.x

### Dataset

* data/data_1.pickle
* data/data_2.pickle

# 1. Environment Setup

Verify the Python, PyTorch and CUDA configuration.

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
Tesla T4


# 2. Clone Repository

Clone the forked repository and access project files.

In [4]:
!git clone https://github.com/Evgeniya-Ryasnova/generative-mocap.git


Cloning into 'generative-mocap'...
remote: Enumerating objects: 252, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 252 (delta 16), reused 0 (delta 0), pack-reused 220 (from 4)
Receiving objects: 100% (252/252), 1.38 GiB | 33.23 MiB/s, done.
Resolving deltas: 100% (37/37), done.
Updating files: 100% (184/184), done.


# 3. Change Working Directory

The working directory is changed to the cloned repository so that Python can access the project scripts, datasets, and output folders correctly.

In [5]:
%cd generative-mocap

/content/generative-mocap


# 4. Install Dependencies

Install required packages.

In [6]:
!pip install spm1d ezc3d statsmodels

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 78.5 MB/s eta 0:00:00


# 5. Verify Training Function

The main training function `train_cgan()` is imported and its function signature is inspected to understand which parameters can be changed for reproduction experiments.

In [ ]:
from train_cgans import train_cgan
import inspect

print(inspect.signature(train_cgan))

/content/generative-mocap/utils.py:486: SyntaxWarning: invalid escape sequence '\%'
  for i in [4, 5, 9]: axs[i].set_xlabel('Gait Cycle [\%]', fontsize=25)
/content/generative-mocap/utils.py:598: SyntaxWarning: invalid escape sequence '\%'
  for ax in axs[2:]: ax.set_xlabel('Gait Cycle [\%]', fontsize=25)


(marker_names=array(['L_IAS', 'L_IPS', 'R_IPS', 'R_IAS', 'R_FTC', 'R_FLE', 'R_FME',
       'R_FAX', 'R_TTC', 'R_FAL', 'R_TAM', 'R_FCC', 'R_FM1', 'R_FM2',
       'R_FM5'], dtype='<U5'), grf_names=array(['force', 'point', 'moment'], dtype='<U6'), ik_names=array(['pelvis_tilt', 'pelvis_list', 'pelvis_rotation', 'hip_flexion_r',
       'hip_adduction_r', 'hip_rotation_r', 'knee_angle_r',
       'ankle_angle_r'], dtype='<U15'), value_cols=['marker_gc', 'grf_3d_gc', 'ik_gc'], names_cols=['marker_names', 'grf_names_3d', 'ik_names'], label_col_contd=['age'], label_col_discr=None, data_df_file=['data/data_1.pickle', 'data/data_2.pickle'], excluded_subjects=[2014001, 2014003, 2015042], z_dim=20, hidden_dim=512, n_epochs=3000, lr=0.002, batch_size=128, display_step=250, n_samples=10, label_contd_lims=[[15, 71, 1]], label_discr_lims=None, model_name='acgan', device=device(type='cuda'), seed=0)


# 6. Test WS-cGAN Training

A short WS-cGAN training run is performed with 10 epochs to verify that the full training pipeline works before running the full 3000-epoch experiment.

## Parameters

- n_epochs = 10
- display_step = 1
- model_name = "wscgan_test"
- condition = walking_speed

In [ ]:
from train_cgans import train_cgan
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_cgan(
    label_col_contd=['walking_speed'],
    label_col_discr=None,
    n_epochs=10,
    display_step=1,
    label_contd_lims=[[0.16, 2.41, 0.02]],
    label_discr_lims=None,
    model_name='wscgan_test',
    device=device,
    seed=0
)

/content/generative-mocap/train_cgans.py:390: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at /pytorch/torch/csrc/tensor/python_tensor.cpp:78.)
  valid = Tensor(real.shape[0], 1).fill_(1.0)


Epoch 1, step 1: Generator loss: 0.06170816719532013, discriminator loss: 0.6723203659057617
Epoch 1, step 2: Generator loss: 0.04478055238723755, discriminator loss: 0.6367135047912598
Epoch 1, step 3: Generator loss: 0.04045671224594116, discriminator loss: 0.650017499923706
Epoch 1, step 4: Generator loss: 0.02108882926404476, discriminator loss: 0.6738141775131226
Epoch 1, step 5: Generator loss: 0.016384847462177277, discriminator loss: 0.6306586265563965
Epoch 1, step 6: Generator loss: 0.012760194949805737, discriminator loss: 0.5233613848686218
Epoch 2, step 7: Generator loss: 0.009743797592818737, discriminator loss: 0.4933582544326782
Epoch 2, step 8: Generator loss: 0.00881848856806755, discriminator loss: 0.47737598419189453
Epoch 2, step 9: Generator loss: 0.00764224398881197, discriminator loss: 0.4228903353214264
Epoch 2, step 10: Generator loss: 0.006682139355689287, discriminator loss: 0.4684189558029175
Epoch 2, step 11: Generator loss: 0.00585491769015789, discrimina

# 7. Check Training Outputs

The output directory is inspected to confirm that the trained generator, synthetic data, and transformation files were saved successfully.

In [ ]:
!ls Results/wscgan_test

back_transform_input.pkl  synthetic_data_after_training.pkl
generator.pt		  transform_labels.pkl


# 8. Notes

### Observations

* The original examples in train_cgans.py are enclosed in triple quotes and are therefore not executed automatically.
* Python 3.12 raises a SyntaxWarning related to '%', but training is unaffected.
* PyTorch 2.x raises deprecation warnings regarding torch.cuda.*Tensor constructors.
* Training completed successfully on an NVIDIA Tesla T4 GPU.

### Current Status

✅ Repository cloned successfully

✅ Dependencies installed successfully

✅ Data loaded successfully

✅ GPU available and functioning

✅ Test WS-cGAN training completed successfully

✅ Synthetic data generated successfully

# 9. Full WS-cGAN Training

The full WS-cGAN model is trained using the original number of epochs from the paper/code.

## Parameters

- n_epochs = 3000
- display_step = 250
- model_name = "wscgan"
- condition = walking_speed
- seed = 0

## Reproducibility settings

Random seed:

- seed = 0

The original implementation uses seed=0. The same value is retained to maximize reproducibility and facilitate comparison with the published results.

In [ ]:
from train_cgans import train_cgan
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_cgan(
    label_col_contd=['walking_speed'],
    label_col_discr=None,
    n_epochs=3000,
    display_step=250,
    label_contd_lims=[[0.16, 2.41, 0.02]],
    label_discr_lims=None,
    model_name='wscgan',
    device=device,
    seed=0
)

Epoch 42, step 250: Generator loss: 0.004184564739931375, discriminator loss: 0.5592460337281228
Epoch 84, step 500: Generator loss: 0.0016667985990643501, discriminator loss: 0.6674943280220031
Epoch 125, step 750: Generator loss: 0.0016533804787322878, discriminator loss: 0.6309146748781205
Epoch 167, step 1000: Generator loss: 0.001345313743688166, discriminator loss: 0.6486919450759888
Epoch 209, step 1250: Generator loss: 0.001446773024275899, discriminator loss: 0.6225510540008545
Epoch 250, step 1500: Generator loss: 0.0015560567290522158, discriminator loss: 0.6034561729431153
Epoch 292, step 1750: Generator loss: 0.0014724970390088857, discriminator loss: 0.6081165008544922
Epoch 334, step 2000: Generator loss: 0.0015714143384248018, discriminator loss: 0.6028767259120941
Epoch 375, step 2250: Generator loss: 0.0015022541233338416, discriminator loss: 0.6009081337451935
Epoch 417, step 2500: Generator loss: 0.001534581586252898, discriminator loss: 0.602676629781723
Epoch 459,

# 10. Check Training Outputs

The output directory is inspected to confirm that the full WS-cGAN training completed successfully and that the generated files were saved correctly.

Expected outputs:

- generator.pt
- synthetic_data_after_training.pkl
- back_transform_input.pkl
- transform_labels.pkl
- entire/
- constrained/
- test_data/

In [ ]:
!ls Results/wscgan

back_transform_input.pkl  synthetic_data_after_training.pkl
constrained		  test_data
entire			  transform_labels.pkl
generator.pt


# 11. Reproduce Results

The evaluation scripts provided in the repository are executed to reproduce the results reported in the original paper.

In [ ]:
!python compare_train_data.py

Spm1d -- Synthetic vs Experimental Data

Conditions          IK         GRF       TOTAL
gcgan          2.8±1.1     1.5±0.4     2.4±0.6
acgan          3.4±3.9     7.9±1.0     4.9±2.7
llcgan         6.3±3.6     1.7±1.5     4.7±2.1
mcgan          8.1±4.4     2.8±2.1     6.3±3.4
wscgan         5.3±3.3     3.5±1.6     4.7±2.1
multicgan      0.5±1.0     7.5±1.0     2.8±0.6
Spm1d -- Synthetic vs Experimental Data

Conditions          IK         GRF       TOTAL
gcgan         18.8±2.8     7.3±1.4    15.0±2.0
acgan         47.8±3.9    13.3±1.6    36.3±2.5
llcgan        22.3±2.4     4.2±3.1    16.2±2.3
mcgan         29.7±7.2     3.7±1.6    21.0±5.0
wscgan        36.0±3.4    16.4±3.3    29.4±3.2
multicgan     10.1±2.8     0.9±0.7     7.0±2.0


In [7]:
!pwd

/content/generative-mocap


In [8]:
!ls

compare_test_data.py			 LICENSE
compare_train_data.py			 plot_results.py
data					 previous_papers.py
datasets.py				 Readme.md
dimensionless_walking_speed_analysis.py  reproduction_colab_Ryasnova.ipynb
dynamical_consistency.py		 Results
environment.yml				 train_cgans.py
Figures					 transformation.py
generative_mocap_reproduction.ipynb	 utils.py


In [9]:
!head -40 compare_test_data.py

# -*- coding: utf-8 -*-
"""

Author:   Metin Bicer
email:    m.bicer19@imperial.ac.uk

Compare synthetic datasets, for the test subjects generated by the conditional
GANs presented in the paper, to the experimental dataset

"""
import numpy as np
from utils import read_dataframes
from utils import get_real_data, get_synthetic_grfm, spmInverse, spmGRF
from utils import np_rmse, np_r_squared, t2test, metrics_m_std
from train_generate import setSeed
import torch
from matplotlib import rc
rc('text', usetex=True)

# feature names (joint angles and moments)
features_ik = np.array(['pelvis_tilt', 'pelvis_list', 'pelvis_rotation',
                        'hip_flexion_r', 'hip_adduction_r', 'hip_rotation_r',
                        'knee_angle_r', 'ankle_angle_r'])
features_grf = np.array(['ground_force_1_vx', 'ground_force_1_vy',
                         'ground_force_1_vz', 'ground_moment_1_my'])
# required variables for getting the experimental data from dataframe
subject_nos = 'all' # all s

In [10]:
!grep -n "def setSeed" train_cgans.py

75:def setSeed(seed=0):


In [11]:
!sed -i 's/from train_generate import setSeed/from train_cgans import setSeed/' compare_test_data.py

In [12]:
!head -20 compare_test_data.py

# -*- coding: utf-8 -*-
"""

Author:   Metin Bicer
email:    m.bicer19@imperial.ac.uk

Compare synthetic datasets, for the test subjects generated by the conditional
GANs presented in the paper, to the experimental dataset

"""
import numpy as np
from utils import read_dataframes
from utils import get_real_data, get_synthetic_grfm, spmInverse, spmGRF
from utils import np_rmse, np_r_squared, t2test, metrics_m_std
from train_cgans import setSeed
import torch
from matplotlib import rc
rc('text', usetex=True)

# feature names (joint angles and moments)


In [14]:
!sed -i "s/rc('text', usetex=True)/rc('text', usetex=False)/" compare_test_data.py

In [15]:
!grep -n "usetex" compare_test_data.py

18:rc('text', usetex=False)


In [16]:
!python compare_test_data.py



2014001
Age: 30, Mass: 67.0 kg
Leg: 862 mm, male
Walking Speeds: 0.27-2.36 m/s

Condition      Speed     IK RMSE        IK R2          IK Sagit RMSE       IK Sagit R2         GRF RMSE       GRF R2         
------------------------------------------------------------------------------------------------------------------------
gcgan          Overall   5.7±1.5        0.64±0.14      7.8±2.8             0.81±0.14           0.07±0.03      0.78±0.13      
acgan          Overall   6.1±1.7        0.65±0.14      8.4±3.0             0.83±0.14           0.07±0.04      0.80±0.13      
llcgan         Overall   6.4±1.3        0.65±0.11      9.2±2.4             0.83±0.11           0.06±0.03      0.81±0.10      
mcgan          Overall   6.4±1.6        0.65±0.14      8.7±3.0             0.82±0.14           0.06±0.04      0.78±0.13      
wscgan         Overall   5.3±1.2        0.75±0.09      6.8±2.3             0.91±0.07           0.05±0.02      0.80±0.12      
multicgan      Overall   5.5±0.9        0

# Note

The WS-cGAN model achieved the best overall performance on the test dataset, showing the lowest IK RMSE (4.7 ± 1.3) and the highest IK R² (0.77 ± 0.09) among all evaluated conditional GAN architectures. These results are consistent with the conclusions reported in the original paper and indicate successful reproduction of the test data evaluation.

In [17]:
!sed -n '65,80p' plot_results.py

    # instantiate figure and axes
    nrows = int(np.ceil(len(plot_feature)/3))
    ncols = int(np.ceil(len(plot_feature)/nrows))
    fig, axs = plt.subplots(nrows, ncols)
    if type(axs) != np.ndarray: axs = np.array(axs)
    axs = axs.flatten()
    # figure sizes
    if nrows == 1 and ncols > 1:
        mngr = plt.get_current_fig_manager()
        mngr.window.setGeometry(38, 125, 1752, 535)
    elif nrows > 1 and ncols > 1:
        mngr = plt.get_current_fig_manager()
        mngr.window.setGeometry(494, 193, 1007, 734)
        #mngr.window.showMaximized()
    # gather mean results for every plot_feature
    mean_trials = np.zeros((len(plot_feature), len(unique_labels), 101))


# Note
The original implementation contains GUI-specific Matplotlib window positioning commands
(mngr.window.setGeometry), which are not supported in Google Colab.

These commands were disabled because they only affect figure placement on a local desktop environment and do not influence the numerical results, statistical analyses, or generated datasets.

In [18]:
!sed -i 's/mngr.window.setGeometry/# mngr.window.setGeometry/g' plot_results.py

In [19]:
!sed -n '65,80p' plot_results.py

    # instantiate figure and axes
    nrows = int(np.ceil(len(plot_feature)/3))
    ncols = int(np.ceil(len(plot_feature)/nrows))
    fig, axs = plt.subplots(nrows, ncols)
    if type(axs) != np.ndarray: axs = np.array(axs)
    axs = axs.flatten()
    # figure sizes
    if nrows == 1 and ncols > 1:
        mngr = plt.get_current_fig_manager()
        # mngr.window.setGeometry(38, 125, 1752, 535)
    elif nrows > 1 and ncols > 1:
        mngr = plt.get_current_fig_manager()
        # mngr.window.setGeometry(494, 193, 1007, 734)
        #mngr.window.showMaximized()
    # gather mean results for every plot_feature
    mean_trials = np.zeros((len(plot_feature), len(unique_labels), 101))


In [23]:
!grep -R "usetex" .

./compare_test_data.py:rc('text', usetex=False)
./dimensionless_walking_speed_analysis.py:rc('text', usetex=False)
./compare_train_data.py:rc('text', usetex=False)


In [25]:
!grep -RIl "usetex=True" . | xargs sed -i "s/usetex=True/usetex=False/g"

sed: no input files


In [26]:
!python dimensionless_walking_speed_analysis.py

hip_flexion_r            RMSE           r              
Very-Slow                0.8            1.0            
Slow                     0.3            1.0            
Free                     0.4            1.0            
Fast                     0.4            1.0            
Very-Fast                0.5            1.0            
knee_angle_r             RMSE           r              
Very-Slow                0.6            1.0            
Slow                     0.4            1.0            
Free                     0.6            1.0            
Fast                     0.7            1.0            
Very-Fast                1.2            1.0            
ankle_angle_r            RMSE           r              
Very-Slow                0.4            1.0            
Slow                     0.4            1.0            
Free                     0.3            1.0            
Fast                     0.3            1.0            
Very-Fast                0.8            1.0     